<div style="max-width:100%;box-sizing:border-box;overflow:visible;border-top:4px solid #0f766e;padding:32px 0 20px;margin:0 0 24px">
  <div style="display:block;color:#0f766e;font-size:13px;line-height:1.8;font-weight:700;letter-spacing:0.8px;text-transform:uppercase;margin:0 0 8px">LAB 2 · DATA WAREHOUSING WITH APACHE DORIS</div>
  <div style="color:#17212b;font-size:30px;line-height:1.3;font-weight:750;margin:0 0 10px">观察存储和写入批次</div>
  <p style="color:#475569;font-size:15px;line-height:1.7;max-width:900px;margin:0">比较同一批订单的两种写入方式，理解数据结果与存储状态。</p>
  <span style="display:inline-block;border:1px solid #99f6e4;border-radius:4px;background:#f0fdfa;color:#115e59;padding:6px 10px;margin-top:14px;font-size:12px">目标 Doris 4.1.3 · 订单数据 · 独立实验库</span>
</div>

完成后，你将比较同一批订单的批量与逐行写入，核对两表均为十行、金额 12220.60，并观察 Tablet 元数据。请按顺序运行。

[讲义](course2_doris_architecture.md) · [课程入口](../README.md)


## 实验范围

仅重建 orders_batch 和 orders_rowwise。使用相同的十笔订单，对照一次写入十行与分十次写入的过程。后台 Compaction 会持续整理数据，观察元数据时请同时记录采样时刻。


In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "dw_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from dw_course.docker_runtime import connect_sandbox
from dw_course.runtime import COURSE_ROOT, fixture, expect, normalized
from dw_course.wwi import HISTORY_COLUMNS as ORDER_COLUMNS, history_ddl as order_ddl, history_rows
from dw_course.ui import show_sql, show_response

lab = connect_sandbox()




## 1. 对齐变量

两张表使用相同的字段、十笔订单、一个桶和单副本。将本会话的 Group Commit 设为 off_mode，逐次观察写入：orders_batch 一次提交十行，orders_rowwise 每次提交一行。

每批写入都有提交与版本管理开销；本步骤用相同业务数据观察批次的影响。


In [ ]:
lab.execute("DROP TABLE IF EXISTS orders_batch")
ddl = order_ddl("orders_batch")
show_sql("建表 SQL", ddl)
lab.execute(ddl)
lab.execute("DROP TABLE IF EXISTS orders_rowwise")
ddl = order_ddl("orders_rowwise")
show_sql("建表 SQL", ddl)
lab.execute(ddl)
rows = history_rows()
lab.execute("SET group_commit = 'off_mode'")
lab.insert("orders_batch", ORDER_COLUMNS, rows)
for row in rows:
    lab.insert("orders_rowwise", ORDER_COLUMNS, [row])


## 2. 查询元数据

SHOW TABLETS 列出表的 Tablet 及版本相关信息。先确认两张表各有一个 Tablet，再比较 VersionCount。它反映采样时的存储版本状态；后台合并已经执行时，两张表的差异可能缩小。


In [ ]:
lab.sql("SHOW CREATE TABLE orders_batch");
lab.sql("SHOW PARTITIONS FROM orders_batch");
lab.sql("SHOW TABLETS FROM orders_batch");
lab.sql("SHOW TABLETS FROM orders_rowwise");


## 3. 先确认业务结果没有变化

两张表都应为 10 行、税前金额 12220.60。先核对明细和汇总，再解释元数据：改变写入批次可以改变物理组织，业务订单及金额应保持一致。


In [ ]:
for table in ("orders_batch", "orders_rowwise"):
    expect(lab.query(f"SELECT COUNT(*), SUM(order_amount) FROM {table}"), [(10, "12220.60")])
lab.sql("EXPLAIN SELECT order_id, order_amount FROM orders_batch WHERE order_id = 1");
lab.sql("SELECT COUNT(*) AS orders, SUM(order_amount) AS amount FROM orders_batch", title="批量写入结果")
lab.sql("SELECT COUNT(*) AS orders, SUM(order_amount) AS amount FROM orders_rowwise", title="逐行写入结果")


## 本实验的观察边界

本实验将业务结果与存储状态分开观察：行数和金额检查数据是否完整，Tablet 元数据帮助理解写入与合并。十行样本用于理解机制；生产性能需要更大数据量、固定并发与持续采样。


## 自己动手

分别对两张表执行只读取 order_id、order_amount 的 EXPLAIN，再加入其他列，找到计划中的输出列变化。EXPLAIN 展示计划，实际扫描量需要执行查询后查看 Query Profile。

再次采样 VersionCount，并结合采样间隔解释结果。


## 独立练习

分别查询两种写入表中第一天的订单数与金额，用一张结果表对照。预期均为 5 笔、3944.20。再观察对应筛选的 EXPLAIN，指出日期过滤条件。

在下一格编写并运行代码，完成后再展开参考解答。空白练习不会被自动判定为完成。


In [ ]:
# 在这里编写你的 SQL 或导入请求。


<details>
<summary>参考解答（完成后再展开）</summary>

```python
query = """SELECT 'batch' AS write_mode, COUNT(*) AS orders, SUM(order_amount) AS amount
FROM orders_batch WHERE order_date = '2013-01-01'
UNION ALL
SELECT 'rowwise', COUNT(*), SUM(order_amount)
FROM orders_rowwise WHERE order_date = '2013-01-01'
ORDER BY write_mode"""
lab.sql(query, title="两种写法的第一天订单")
expect(lab.query(query), [("batch", 5, "3944.20"), ("rowwise", 5, "3944.20")])
lab.sql("EXPLAIN SELECT order_id, order_amount FROM orders_batch WHERE order_date = '2013-01-01'", title="筛选计划")
```

</details>
